In [ ]:
from openai import OpenAI
from IPython.display import Markdown
import json, os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or None
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or None
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or None

In [ ]:
PROVIDERS = {
    "openrouter": [OPENROUTER_BASE_URL, OPENROUTER_API_KEY],
    "gemini": [GEMINI_BASE_URL, GEMINI_API_KEY],
    "groq": [GROQ_BASE_URL, GROQ_API_KEY]
}

In [ ]:
if GEMINI_API_KEY:
  print("GEMINI_API_KEY available and looks good so far")
  print(f"KEY start with {GEMINI_API_KEY[:5]}")
else:
  print("No Gemini API KEY Found or connected.")


if OPENROUTER_API_KEY and OPENROUTER_API_KEY.startswith("sk-or-v1-"):
  print("\n\nOPENROUTER_KEY is available and looks good so far")
else:
  print("No OPENROUTER_API_KEY is found or connected.")

In [ ]:
gemini_client = OpenAI(base_url=GEMINI_BASE_URL, api_key=GEMINI_API_KEY)
openrouter_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)
groq_client = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
available_clients = {
    "operouter": openrouter_client,
    "gemini_client": gemini_client,
    "groq_client": groq_client,
    "local-ollama": ollama_client
}

### Function Tool to generate random dataset with filler values

In [ ]:
def generate_random_user_dataset(k: int = 1) -> str:
    import requests as re

    result = re.get(f"https://randomuser.me/api/?results={k}")
    user_data = result.json()

    print(f"Randomuser.me Response: {user_data}")

    return user_data

In [ ]:
generate_random_user_dataset()

### A helper function to print the current messages and roles

In [ ]:
available_functions = {
    "generate_random_topic_dataset": generate_random_user_dataset
}

In [ ]:
SYSTEM_PROMPT = """
You are an expert Data Synthesis Agent specializing in creating realistic mock data for software testing and prototyping.
You have access to the `generate_random_user_dataset` tool, which retrieves realistic random human identities.

Follow these strict operating guidelines:
1. Tool Triggering: Whenever a user requests synthetic data for people (e.g., employees, customers, patients, users), immediately call the `generate_random_user_dataset` tool. Pass the requested number of records as the parameter `k`.
2. Context Adaptation: The tool returns generic identity data (names, emails, locations). You must adapt this raw data to fit the user's specific prompt. For example, if the user asks for "Employees at XYZ Company," extract the names/locations from the tool, but creatively invent missing contextual fields (like "Job Title", "Department", or "Hire Date") to fulfill the prompt.
3. Data Presentation: Your final response MUST be a clean, well-structured Markdown table containing the 5-8 most relevant fields (e.g., Full Name, Email, Location, + your synthesized contextual fields).
4. Strict Output Constraint: Do not include any conversational filler, greetings, or explanations before or after the table. Your response must begin exactly with the first character of the Markdown table (e.g., `|`).
"""

In [ ]:
USER_MESSAGE = "Generate a synthetic dataset of 10 Employees for some random XYZ company with accurate real-life attributes using the predefined Tools"

In [ ]:
MODEL_CONFIG = {
    "gemini": {
        "3.5": {
            "flash-lite": "gemini-3.5-flash-lite",
            "flash": "gemini-3.5-flash",
            "pro": "gemini-3.5-pro",
        },
        "3.6": {
            "flash": "gemini-3.6-flash",
            "pro": "gemini-3.6-pro",
        },
        "3.7": {
            "flash": "gemini-3.7-flash",
        },
        "3.8": {
            "flash": "gemini-3.8-flash",
        }
    },
    "openrouter": {
        "nvidia": {
            "ultra": "nvidia/nemotron-3-ultra-550b-a55b:free",
            "lightning": "nvidia/nemotron-3.5-lightning:free",
            "super": "nvidia/nemotron-3-super-120b-a12b:free",
            "nano-omni": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
        },
        "free": "openrouter/free",
    },
    "groq": {
        "llama": {
            "3.1": "llama-3.1-8b-instant",
            "3.3": "llama-3.3-70b-versatile"
        },
        "openai": {
            "oss-120b": "openai/gpt-oss-120b",
            "oss-20b": "openai/gpt-oss-20b",
        },
        "qwen": {
            "3.8": "qwen/qwen3.8-27b"
        }
    }
}


In [ ]:
MESSAGES = [
    {'role': "system", 'content': SYSTEM_PROMPT}
]

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "generate_random_user_dataset",
            "description": "Generates a mock JSON dataset of `k` no. of users.",
            "parameters": {
                "k": {
                    "type": "number",
                    "description": "The number of rows/records to generate. Keep small for context window limits."
                }
            }
        }
    }
]

In [ ]:
MODEL = MODEL_CONFIG['groq']['openai']['oss-20b']

In [ ]:
import json
from openai import OpenAI

In [ ]:

def clean_assistant_message(msg) -> dict:
    """Keep only the fields the API accepts (drops reasoning, etc.)."""
    clean = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        clean["tool_calls"] = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ]
    return clean

In [ ]:

def generate_dataset(client: OpenAI, user_message: str = "Generate a dataset of 5 rows"):
    # Local copy: global MESSAGES (system prompt etc.) is never mutated
    messages = list(MESSAGES)
    messages.append({"role": "user", "content": user_message})

    for _ in range(5):  # max tool-calling rounds, prevents infinite loops
        res = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
        )
        message = res.choices[0].message

        # No tool calls -> this is the final answer
        if not message.tool_calls:
            return message.content

        # Append a CLEAN assistant message (no reasoning fields)
        messages.append(clean_assistant_message(message))

        for tool_call in message.tool_calls:
            try:
                args = json.loads(tool_call.function.arguments)
                result = generate_random_user_dataset(**args)
            except Exception as e:
                result = {"error": str(e)}  # let the model see the failure

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

    return "⚠️ The agent did not finish within the allowed number of tool calls."

In [ ]:
user_data = generate_dataset(groq_client, "Generate me a dataset of about 10 users")

In [ ]:
Markdown(user_data)

# Complete workflow using Gradio Elements

In [ ]:
import csv
import re
import tempfile
import time
import os
import gradio as gr

In [ ]:
def gradio_generate_dataset(user_message):
    result = generate_dataset(groq_client, user_message)
    return result

In [ ]:
def generate_with_feedback(message):
    if not message or not message.strip():
        yield "⚠️ Please enter a prompt first.", "", gr.update(interactive=True, value="Generate")
        return

    yield "⏳ Agent is working...", "", gr.update(interactive=False, value="Generating...")

    start = time.time()
    try:
        result = gradio_generate_dataset(message)
        elapsed = time.time() - start
        yield f"✅ Done in {elapsed:.1f}s.", result, gr.update(interactive=True, value="Generate")
    except Exception as e:
        yield f"❌ Something went wrong: {e}", "", gr.update(interactive=True, value="Generate")


In [ ]:
def extract_rows(text: str):
    """Pull table rows out of a Markdown table or a ```csv code block."""
    # 1. Markdown table
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("|"):
            continue
        cells = [c.strip() for c in line.strip("|").split("|")]
        if all(re.fullmatch(r":?-+:?", c) for c in cells):  # separator row
            continue
        rows.append(cells)
    if len(rows) > 1:
        return rows

    # 2. Fallback: ```csv ... ``` code block
    match = re.search(r"```(?:csv)?\s*\n(.*?)```", text, re.S)
    if match:
        parsed = list(csv.reader(match.group(1).strip().splitlines()))
        if len(parsed) > 1:
            return parsed

    return None

In [ ]:
def save_csv(text: str):
    """Write the extracted rows to a temp CSV file. Returns the path, or None."""
    rows = extract_rows(text)
    if not rows:
        return None
    path = os.path.join(tempfile.mkdtemp(), "synthetic_dataset.csv")
    with open(path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)
    return path

In [ ]:
def generate_with_feedback(message):
    hide_download = gr.update(visible=False, value=None)

    if not message or not message.strip():
        yield "⚠️ Please enter a prompt first.", "", gr.update(interactive=True, value="Generate"), hide_download
        return

    yield "⏳ Agent is working...", "", gr.update(interactive=False, value="Generating..."), hide_download

    start = time.time()
    try:
        result = gradio_generate_dataset(message)
        elapsed = time.time() - start

        csv_path = save_csv(result)
        if csv_path:
            download = gr.update(value=csv_path, visible=True)
            status_msg = f"✅ Done in {elapsed:.1f}s. You can download the CSV below."
        else:
            download = hide_download
            status_msg = f"✅ Done in {elapsed:.1f}s. (No table found, so CSV download isn't available.)"

        yield status_msg, result, gr.update(interactive=True, value="Generate"), download
    except Exception as e:
        yield f"❌ Something went wrong: {e}", "", gr.update(interactive=True, value="Generate"), hide_download

In [ ]:

with gr.Blocks() as demo:
    gr.Markdown("# 📊 Synthetic Data Generator Agent")
    gr.Markdown("Enter a prompt requesting user/employee data, and the agent will build a synthetic dataset for you.")

    user_input = gr.Textbox(
        label="User Message",
        placeholder="e.g., Generate a synthetic dataset of 10 Employees for some random XYZ company...",
        lines=3,
    )

    gr.Examples(
        examples=[
            "Generate a synthetic dataset of 10 employees for a random XYZ company.",
            "Create 15 users with name, email, age and city.",
            "Make a dataset of 8 software engineers with salary and department.",
        ],
        inputs=user_input,
    )

    submit_btn = gr.Button("Generate", variant="primary")

    status = gr.Markdown()
    output_markdown = gr.Markdown(label="Synthesized Dataset")
    download_btn = gr.DownloadButton("⬇️ Download CSV", visible=False)

    outputs = [status, output_markdown, submit_btn, download_btn]
    submit_btn.click(fn=generate_with_feedback, inputs=user_input, outputs=outputs)
    user_input.submit(fn=generate_with_feedback, inputs=user_input, outputs=outputs)

In [ ]:
demo.queue()
demo.launch(debug=True, inbrowser=True, share=True)

In [ ]:
demo.launch(debug=True)